# ML-09 — Validation and Research Claim Audit

This notebook audits the Week-5 model using an honest client-grouped split, checks leakage, reviews failure examples, and rewrites the final claim carefully.

> If `is_declining_label` is unavailable locally, `trend_direction == 'down'` is used only as an evaluation-only proxy and is never used as a feature.

## 1. Two paper findings + my methodology questions

**Finding 1 — Content lifecycle / age.** The FlyRank report observes that growing pages were younger on average (about 185 days) than declining pages (about 228 days), while word count was nearly the same at roughly 1.5K words. **Methodology question:** could client/site mix, topic, historical visibility, or update history explain part of this association? I would treat age as an observed signal rather than evidence that age itself causes decline.

**Finding 2 — Freshness multiplier.** The report observes a 5.43:1 growth-to-decline ratio in the 31–90 day freshness window and separately reports a large difference between refreshed and stale pages in the 365+ cohort. It also notes that the 361+ bucket contains only 21 declining pages, so that bucket is unstable. **Methodology question:** were refreshed and stale pages comparable before the refresh, or could selection effects explain part of the difference? A stronger validation design would control for baseline performance, client/site mix, and pre-refresh characteristics.

These are useful directional observations, but they do not by themselves establish causality.

In [ ]:
from pathlib import Path
import subprocess
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import recall_score, precision_score, f1_score, roc_auc_score, confusion_matrix

repo_root = Path('/content/ml-internship-2026')
if not repo_root.exists():
    subprocess.run(['git','clone','https://github.com/engyusufayman06/ml-internship-2026.git',str(repo_root)], check=True)
data_path = repo_root / 'data/raw/content_refresh_anonymized.csv'
if not data_path.exists():
    raise FileNotFoundError('Dataset not found. Make the warehouse data available before Run All.')
df = pd.read_csv(data_path)
print('Shape:', df.shape)

In [ ]:
target = 'is_declining_label'
if target not in df.columns:
    df[target] = (df['trend_direction'].astype(str).str.lower() == 'down').astype(int)
    print('Using trend_direction==down as an evaluation-only proxy label.')
forbidden = {target,'trend_direction','trend_pct','content_id','client_id','score','reason_code','action_label','freshness_bucket','volume_bucket'}
features = [c for c in df.columns if c not in forbidden]
X, y = df[features].copy(), df[target].astype(int)
groups = df['client_id'].astype(str)
num = X.select_dtypes(include=np.number).columns.tolist()
cat = [c for c in X.columns if c not in num]
prep = ColumnTransformer([('num',Pipeline([('imp',SimpleImputer(strategy='median',add_indicator=True)),('scale',StandardScaler())]),num),('cat',Pipeline([('imp',SimpleImputer(strategy='most_frequent')),('ohe',OneHotEncoder(handle_unknown='ignore'))]),cat)])
def make_model():
    return Pipeline([('prep',prep),('model',LogisticRegression(max_iter=1000,class_weight='balanced',random_state=42))])
print('Features:',len(features),'| Positive rate:',round(y.mean(),4))

## 2. My model under an honest split (before/after)

The **before** result uses a random 80/20 split. The **after** result uses `GroupShuffleSplit` by `client_id`, so no client appears in both train and test. The model, preprocessing, features, seed, and metrics remain the same.

In [ ]:
def p_at_k(y_true, score, k):
    k=min(k,len(y_true)); order=np.argsort(-np.asarray(score))[:k]
    return float(np.asarray(y_true)[order].mean())
def evaluate(train_idx,test_idx,name):
    m=make_model(); m.fit(X.iloc[train_idx],y.iloc[train_idx])
    prob=m.predict_proba(X.iloc[test_idx])[:,1]; pred=(prob>=0.5).astype(int); yt=y.iloc[test_idx].to_numpy()
    return {'split':name,'Recall':recall_score(yt,pred,zero_division=0),'Precision':precision_score(yt,pred,zero_division=0),'F1':f1_score(yt,pred,zero_division=0),'ROC-AUC':roc_auc_score(yt,prob),'Precision@20':p_at_k(yt,prob,20),'Precision@50':p_at_k(yt,prob,50),'model':m,'prob':prob,'pred':pred,'test_idx':np.asarray(test_idx)}
idx=np.arange(len(df))
tr_r,te_r=train_test_split(idx,test_size=0.20,random_state=42,stratify=y)
gss=GroupShuffleSplit(n_splits=1,test_size=0.20,random_state=42)
tr_g,te_g=next(gss.split(X,y,groups=groups))
before=evaluate(tr_r,te_r,'Before: random 80/20')
after=evaluate(tr_g,te_g,'After: client-grouped 80/20')
rows=[{k:v for k,v in before.items() if k not in {'model','prob','pred','test_idx'}},{k:v for k,v in after.items() if k not in {'model','prob','pred','test_idx'}}]
print(pd.DataFrame(rows).to_string(index=False,float_format=lambda x:f'{x:.4f}'))
print('Random client overlap:',len(set(groups.iloc[tr_r]) & set(groups.iloc[te_r])))
print('Grouped client overlap:',len(set(groups.iloc[tr_g]) & set(groups.iloc[te_g])))
assert len(set(groups.iloc[tr_g]) & set(groups.iloc[te_g]))==0

In [ ]:
test_df=df.iloc[te_g].copy(); yt=y.iloc[te_g].to_numpy()
flag=(test_df['days_since_last_update']>=180) & (test_df['impressions_90d']>=3000)
base_pred=flag.astype(int).to_numpy(); base_score=test_df['impressions_90d'].where(flag,0).to_numpy()
baseline={'split':'W04 baseline on grouped test','Recall':recall_score(yt,base_pred,zero_division=0),'Precision':precision_score(yt,base_pred,zero_division=0),'F1':f1_score(yt,base_pred,zero_division=0),'ROC-AUC':roc_auc_score(yt,base_score),'Precision@20':p_at_k(yt,base_score,20),'Precision@50':p_at_k(yt,base_score,50)}
print(pd.DataFrame([baseline,{k:v for k,v in after.items() if k not in {'model','prob','pred','test_idx'}}]).to_string(index=False,float_format=lambda x:f'{x:.4f}'))

## 3. Leakage audit

The final feature set excludes the target, fields used to derive the target, identifiers, and fields generated by the Week-4 decision rule.

In [ ]:
leakage=['is_declining_label','trend_direction','trend_pct','content_id','client_id','score','reason_code','action_label','freshness_bucket','volume_bucket']
audit=pd.DataFrame({'field':leakage,'present_in_data':[f in df.columns for f in leakage],'used_as_feature':[f in features for f in leakage]})
print(audit.to_string(index=False))
assert not audit.used_as_feature.any()
print('LEAKAGE AUDIT PASSED')

## Real failure examples

These examples come from the grouped holdout. IDs remain pseudonymized. False negatives are declining pages missed by the model; false positives are pages flagged by the model that were not labeled declining.

In [ ]:
err=test_df.copy(); err['actual']=yt; err['predicted']=after['pred']; err['probability']=after['prob']
safe=[c for c in ['content_id','client_id','content_type','days_since_last_update','impressions_90d'] if c in err.columns]
fn=err[(err.actual==1)&(err.predicted==0)].sort_values('probability').head(3)
fp=err[(err.actual==0)&(err.predicted==1)].sort_values('probability',ascending=False).head(3)
print('FALSE NEGATIVES'); print(fn[safe+['actual','predicted','probability']].to_string(index=False))
print('\nFALSE POSITIVES'); print(fp[safe+['actual','predicted','probability']].to_string(index=False))
print('\nConfusion matrix:'); print(confusion_matrix(yt,after['pred']))

In [ ]:
m=after['model']; names=m.named_steps['prep'].get_feature_names_out(); coef=m.named_steps['model'].coef_[0]
coef_df=pd.DataFrame({'feature':names,'coefficient':coef})
print('Top positive coefficients'); print(coef_df.nlargest(10,'coefficient').to_string(index=False))
print('\nTop negative coefficients'); print(coef_df.nsmallest(10,'coefficient').to_string(index=False))

## 4. Claim rewrite

**Too strong:** The Logistic Regression model accurately identifies declining content and can improve the content-refresh process.

**Rewritten:** On the evaluated client-grouped holdout, Logistic Regression produced measured recall, precision, F1, ROC-AUC, and Precision@K results. These results provide directional evidence that the available signals can support a review-prioritization workflow; they do not establish causality or guarantee performance on future clients or time periods.

**Operational interpretation:** the model is decision-support for human review, not an automatic decision-maker.

## Self-check

- [x] Required sections are filled with reasoning and supporting code
- [ ] Run the notebook top-to-bottom in Colab before submission
- [x] No client names or private queries are included
- [x] Claims use careful language: observed, measured, directional, decision-support
- [ ] Commit the executed notebook with outputs under `work/notebooks/w06_validation_audit.ipynb`